In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/jonathantran/Desktop/nba_economics")
CONTRACTS_DIR = PROJECT_ROOT / "data/staging/contracts"

files = [
    "s11a_rookie_salary_scale.csv",
    "s11b_salary_min_final.csv"
]

for filename in files:
    df = pd.read_csv(CONTRACTS_DIR / filename)

    print(f"\nFILE: {filename}")
    print("SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    print("MISSING VALUES:")
    print(df.isna().sum().to_dict())
    print("DUPLICATE ROWS:", df.duplicated().sum())
    print("SEASONS:", df["season"].nunique() if "season" in df else "No season column")

    display(df.head(5))


FILE: s11a_rookie_salary_scale.csv
SHAPE: (775, 8)
COLUMNS: ['draft_pick', 'year_1_salary', 'year_2_salary', 'year_3_option_salary', 'year_4_option_increase_pct', 'qualifying_offer_increase_pct', 'season', 'year']
MISSING VALUES:
{'draft_pick': 0, 'year_1_salary': 0, 'year_2_salary': 0, 'year_3_option_salary': 0, 'year_4_option_increase_pct': 0, 'qualifying_offer_increase_pct': 0, 'season': 0, 'year': 0}
DUPLICATE ROWS: 0
SEASONS: 26


,draft_pick,year_1_salary,year_2_salary,year_3_option_salary,year_4_option_increase_pct,qualifying_offer_increase_pct,season,year
0,1,11521600,12097900,12673900,0.261,0.400,2025-2026,2026
1,2,10308600,10824200,11339800,0.262,0.405,2025-2026,2026
2,3,9257400,9719900,10183200,0.264,0.412,2025-2026,2026
3,4,8346400,8763800,9181300,0.265,0.419,2025-2026,2026
4,5,7558200,7935900,8313800,0.267,0.426,2025-2026,2026



FILE: s11b_salary_min_final.csv
SHAPE: (286, 5)
COLUMNS: ['season', 'year', 'cba_id', 'years_of_service', 'minimum_salary']
MISSING VALUES:
{'season': 0, 'year': 0, 'cba_id': 0, 'years_of_service': 0, 'minimum_salary': 0}
DUPLICATE ROWS: 0
SEASONS: 26


,season,year,cba_id,years_of_service,minimum_salary
0,2011-2012,2012,cba_2011,0,473604
1,2011-2012,2012,cba_2011,1,762195
2,2011-2012,2012,cba_2011,2,854389
3,2011-2012,2012,cba_2011,3,885120
4,2011-2012,2012,cba_2011,4,915852


In [4]:
from pathlib import Path
import pandas as pd

CANONICAL_DIR = PROJECT_ROOT / "data/canonical"

rookie = pd.read_csv(
    CONTRACTS_DIR / "s11a_rookie_salary_scale.csv"
)

minimum = pd.read_csv(
    CONTRACTS_DIR / "s11b_salary_min_final.csv"
)

seasons = pd.read_csv(
    CANONICAL_DIR / "seasons.csv"
)

# 1. Check season coverage.
for name, df in [("ROOKIE SCALE", rookie), ("MINIMUM SALARY", minimum)]:
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Season range:", sorted(df["season"].unique())[0],
          "to", sorted(df["season"].unique())[-1])

    print("Missing canonical seasons:",
          sorted(set(seasons["season"]) - set(df["season"])))

# 2. Check source grain and numeric validity.
assert not rookie.duplicated(["season", "draft_pick"]).any()
assert not minimum.duplicated(["season", "years_of_service"]).any()

assert rookie["season"].isin(seasons["season"]).all()
assert minimum["season"].isin(seasons["season"]).all()

assert rookie["draft_pick"].between(1, 30).all()
assert rookie[
    ["year_1_salary", "year_2_salary", "year_3_option_salary"]
].ge(0).all().all()

assert minimum["minimum_salary"].ge(0).all()

# Validate service-year categories without changing their meaning.
valid_service_years = {str(i) for i in range(10)} | {"10+"}

assert minimum["years_of_service"].astype(str).isin(
    valid_service_years
).all(), "Unexpected years-of-service category"

# 3. Verify minimum-salary CBA identifiers against C01.
cba_check = minimum.merge(
    seasons[["season", "cba_id"]],
    on="season",
    how="left",
    validate="many_to_one",
    suffixes=("_source", "_canonical")
)

cba_mismatches = cba_check[
    cba_check["cba_id_source"] != cba_check["cba_id_canonical"]
]

print("\nCBA mismatches:", len(cba_mismatches))
display(cba_mismatches.head(10))

assert cba_mismatches.empty, "CBA identifiers do not match C01."

# 4. Show draft-pick coverage by season.
print("\nROOKIE SCALE PICK COVERAGE")
display(
    rookie.groupby("season")["draft_pick"]
    .agg(["count", "min", "max"])
    .sort_index()
)

# 5. Export both physical tables.
rookie = rookie.sort_values(
    ["season", "draft_pick"]
).reset_index(drop=True)

minimum = minimum.sort_values(
    ["season", "years_of_service"]
).reset_index(drop=True)

rookie.to_csv(
    CANONICAL_DIR / "rookie_salary_scale.csv",
    index=False
)

minimum.to_csv(
    CANONICAL_DIR / "minimum_salary_rules.csv",
    index=False
)

print("\nC09 VALIDATION PASSED")
print("---------------------------------------------")
print("Rookie salary scale rows:", len(rookie))
print("Minimum salary rule rows:", len(minimum))
print("Both files exported successfully.")


ROOKIE SCALE
Rows: 775
Season range: 2000-2001 to 2025-2026
Missing canonical seasons: []

MINIMUM SALARY
Rows: 286
Season range: 2000-2001 to 2025-2026
Missing canonical seasons: []

CBA mismatches: 0


,season,year,cba_id_source,years_of_service,minimum_salary,cba_id_canonical



ROOKIE SCALE PICK COVERAGE


,count,min,max
season,,,
2000-2001,29,1,29
2001-2002,29,1,29
2002-2003,29,1,29
2003-2004,29,1,29
2004-2005,29,1,29
2005-2006,30,1,30
2006-2007,30,1,30
2007-2008,30,1,30
2008-2009,30,1,30



C09 VALIDATION PASSED
---------------------------------------------
Rookie salary scale rows: 775
Minimum salary rule rows: 286
Both files exported successfully.


In [3]:
print(minimum["years_of_service"].unique().tolist())
print(minimum["years_of_service"].dtype)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10+']
str
